# LTV PostgreSQL Analysis

This notebook performs business-focused SQL analysis on the final customer LTV and high-priority customer tables stored in PostgreSQL.

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from urllib.parse import quote_plus
import os

In [2]:
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = quote_plus(os.getenv("DB_PASSWORD"))
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Database connection created successfully.")

Database connection created successfully.


In [3]:
pd.read_sql(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public';
    """,
    engine
)

,table_name
0,telco_customer_churn
1,customer_churn_ltv
2,high_priority_customers


In [5]:
pd.read_sql(
    "SELECT COUNT(*) AS total_customers FROM customer_churn_ltv;",
    engine
)

,total_customers
0,7032


In [4]:
pd.read_sql(
    "SELECT COUNT(*) AS high_priority_customers FROM high_priority_customers;",
    engine
)

,high_priority_customers
0,254


*Both PostgreSQL tables were verified successfully. The main customer analytics table contains all customers, while the high-priority table contains only customers classified as High Value - High Risk.*

In [6]:
# Customer count by ltv segment
pd.read_sql(
    """
    SELECT 
        "LTV_Segment",
        COUNT(*) AS customer_count
    FROM customer_churn_ltv
    GROUP BY "LTV_Segment"
    ORDER BY customer_count DESC;
    """,
    engine
)

,LTV_Segment,customer_count
0,Low Value,3516
1,High Value,1758
2,Medium Value,1758


In [7]:
# Average LTV by customer priority
pd.read_sql(
    """
    SELECT 
        "Customer_Priority",
        COUNT(*) AS customer_count,
        ROUND(AVG("Estimated_LTV")::numeric, 2) AS average_ltv
    FROM customer_churn_ltv
    GROUP BY "Customer_Priority"
    ORDER BY average_ltv DESC;
    """,
    engine
)

,Customer_Priority,customer_count,average_ltv
0,High Value - Low Risk,1504,5747.11
1,High Value - High Risk,254,5534.91
2,Medium Value - High Risk,405,2422.89
3,Medium Value - Low Risk,1353,2393.33
4,Low Value - Low Risk,2306,568.38
5,Low Value - High Risk,1210,392.93


In [8]:
# High-priority customers summary
pd.read_sql(
    """
    SELECT 
        COUNT(*) AS high_priority_customers,
        ROUND(AVG("Estimated_LTV")::numeric, 2) AS average_ltv,
        ROUND(SUM("Estimated_LTV")::numeric, 2) AS total_ltv_at_risk
    FROM high_priority_customers;
    """,
    engine
)

,high_priority_customers,average_ltv,total_ltv_at_risk
0,254,5534.91,1405866.15


In [9]:
# Top 10 high-priority customers by LTV
pd.read_sql(
    """
    SELECT 
        "Estimated_LTV",
        "MonthlyCharges",
        "tenure",
        "Customer_Priority"
    FROM high_priority_customers
    ORDER BY "Estimated_LTV" DESC
    LIMIT 10;
    """,
    engine
)

,Estimated_LTV,MonthlyCharges,tenure,Customer_Priority
0,8481.60,117.80,72,High Value - High Risk
1,8095.50,115.65,70,High Value - High Risk
2,8088.50,115.55,70,High Value - High Risk
3,7994.00,114.20,70,High Value - High Risk
4,7929.45,118.35,67,High Value - High Risk
5,7866.00,109.25,72,High Value - High Risk
6,7785.40,116.20,67,High Value - High Risk
7,7710.60,108.60,71,High Value - High Risk
8,7694.20,113.15,68,High Value - High Risk
9,7671.55,108.05,71,High Value - High Risk
